# Q2 · Unsupervised Learning — Customer Segmentation
**Notebook:** `q2_unsupervised.ipynb`  
**Dataset:** `../data/q2_customers.csv`  
**Goal:** Customer segmentation using K-Means clustering; visualise results with PCA.  
**Columns:** `age`, `annual_spend`, `visits_per_month`, `basket_size`, `days_since_last_visit`, `num_categories_purchased`  
**Author:** Gaurav Anand Shukla | BITSoM_BA_25111017

## Task 1 · Data Preparation — 3 marks

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/q2_customers.csv')
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
print('\nDescriptive Statistics:')
print(df.describe().round(2))

Shape: (500, 6)

Data Types:
age                         int64
annual_spend                int64
visits_per_month            int64
basket_size                 int64
days_since_last_visit       int64
num_categories_purchased    int64
dtype: object

Missing values: 0

Descriptive Statistics:
           age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
count  500.00        500.00            500.00       500.00                 500.00                    500.00
mean    40.45      48856.95              8.41      2682.29                  49.47                      4.66
std     14.43      32856.80              5.32      2274.96                  49.70                      2.44
min     18.00       5038.00              1.00       212.00                   0.00                      1.00
25%     28.00      19213.25              4.00       727.75                  12.00                      3.00
50%     41.00      44257.00              8.00      2051.50  

In [2]:
# Scale all features with StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)

print('After StandardScaler — first 3 rows:')
print(X_scaled_df.head(3).round(4))
print('\nMean per feature (should be ~0):', X_scaled.mean(axis=0).round(4))
print('Std  per feature (should be ~1):', X_scaled.std(axis=0).round(4))

After StandardScaler — first 3 rows:
        age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
0 -0.7248       -0.1759            0.1107       -0.2648                -0.0899                    0.5491
1 -1.4917       -1.0407            0.4947       -0.9807                -0.8362                   -0.6801
2  0.1729       -0.1681           -0.4534       -0.2824                -0.6772                   -0.2655

Mean per feature (should be ~0): [ 0.  0. -0. -0.  0.  0.]
Std  per feature (should be ~1): [1. 1. 1. 1. 1. 1.]


### ✅ Why Scaling is Essential Before K-Means
**K-Means uses Euclidean distance** to assign points to the nearest centroid. Without scaling:
- `annual_spend` ranges from ₹5,038–₹119,757 while `visits_per_month` ranges from 1–19.
- Without scaling, `annual_spend` would completely **dominate the distance calculation**, making all other features irrelevant.
- After `StandardScaler`, every feature has **mean=0 and std=1**, ensuring each variable contributes equally to cluster assignments.
- This produces meaningful, interpretable cluster centroids that reflect true behavioural patterns across all dimensions.

## Task 2 · Choosing K — Elbow Method — 5 marks

In [3]:
# Compute WCSS for K = 1 through 10
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

print('WCSS values for K=1 to K=10:')
for k, w in zip(K_range, wcss):
    print(f'  K={k:2d}:  WCSS = {w:.2f}')

WCSS values for K=1 to K=10:
  K= 1:  WCSS = 3000.00
  K= 2:  WCSS = 968.99
  K= 3:  WCSS = 561.25
  K= 4:  WCSS = 444.93
  K= 5:  WCSS = 402.37
  K= 6:  WCSS = 370.39
  K= 7:  WCSS = 346.95
  K= 8:  WCSS = 319.90
  K= 9:  WCSS = 303.28
  K=10:  WCSS = 289.11


In [4]:
# Plot Elbow Curve
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(K_range, wcss, 'bo-', linewidth=2.5, markersize=8)
ax.fill_between(K_range, wcss, alpha=0.1)
ax.axvline(x=4, color='#E74C3C', linestyle='--', linewidth=2, label='Optimal K=4 (Elbow)')
ax.scatter([4], [wcss[3]], color='#E74C3C', s=200, zorder=5)
ax.set_title('Elbow Method — Within-Cluster Sum of Squares (WCSS)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('WCSS (Inertia)')
ax.set_xticks(K_range)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('elbow_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('Elbow curve plotted and saved.')

Elbow curve plotted and saved.


### 📊 Optimal K — Justification
From the WCSS values and elbow plot, **K=4 is the optimal number of clusters**:
- **K=1→2**: WCSS drops steeply by 2031 points (68% reduction) — massive gain.
- **K=2→3**: Drops by 408 points (42% reduction) — still substantial.
- **K=3→4**: Drops by 116 points (21% reduction) — meaningful improvement.
- **K=4→5**: Drops by only 42 points (9.5%) — the rate of improvement sharply slows — this is the **elbow**.
- **Business rationale**: 4 customer segments is practically manageable — a marketing team can design 4 distinct strategies. Fewer segments (3) would merge meaningfully different groups; more (5+) creates overlap and complexity.

## Task 3 · K-Means Clustering — 6 marks

In [5]:
# Fit K-Means with optimal K=4
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10, random_state=42)
kmeans.fit(X_scaled)

# Add cluster labels to dataframe
df['cluster'] = kmeans.labels_

print('Cluster assignment counts:')
print(df['cluster'].value_counts().sort_index())

Cluster assignment counts:
cluster
0    170
1     80
2    165
3     85
dtype: int64


In [6]:
# Print cluster centroids in original (interpretable) scale
centroids_scaled   = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids_original, columns=df.columns[:-1])
centroids_df.index.name = 'Cluster'
print('Cluster Centroids (original scale):')
print(centroids_df.round(1).to_string())

Cluster Centroids (original scale):
           age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
Cluster
0         24.7       14847.4              14.3        559.0                    9.1                       2.1
1         57.0       89814.1               2.5       5296.4                  148.0                       7.5
2         40.4       43340.7               8.2       2021.7                   35.2                       4.4
3         56.5       89036.2               2.6       5751.0                   65.2                       7.5


### 🧩 Cluster Business Interpretation

| Cluster | Size | Label | Key Characteristics | Recommended Strategy |
|---------|------|-------|--------------------|-----------------------|
| **0** | 170 | 🔵 Young Frequent Shoppers | Youngest (25y), lowest spend (₹14.8K), highest visit frequency (14/mo), very small baskets, recent visitors | Upsell bundles, app loyalty rewards, small-ticket promotions |
| **1** | 80  | 🟠 Lapsed High-Value Seniors | Oldest (57y), very high spend (₹89.8K), very low frequency (2.5/mo), large baskets, last visited 148 days ago | Win-back campaigns, personalised premium offers, VIP outreach |
| **2** | 165 | 🟢 Core Mid-Market Regulars | Mid-age (40y), moderate spend (₹43.3K), regular visits (8/mo), average basket | Mainstream promotions, cross-sell adjacent categories |
| **3** | 85  | 🔴 Engaged Premium Loyalists | Older (57y), very high spend (₹89K), very low frequency (2.6/mo), largest baskets (₹5,751), recent | Priority loyalty programme, early product access, high-value retention |

**Key Insight:** Clusters 1 and 3 are both high-spending older customers, separated by **recency** — Cluster 1 is lapsed (148 days) and needs re-engagement, while Cluster 3 is still active (65 days). These two groups require completely different marketing approaches despite similar spend profiles.

## Task 4 · Dimensionality Reduction with PCA — 5 marks

In [7]:
# Apply PCA to reduce to 2 principal components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print('Explained Variance Ratio per Component:')
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {ratio:.4f}  ({ratio*100:.2f}%)')
print(f'\nTotal Variance Explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.2f}%')

Explained Variance Ratio per Component:
  PC1: 0.8356  (83.56%)
  PC2: 0.0557  (5.57%)

Total Variance Explained by 2 PCs: 89.13%


In [8]:
# Feature loadings (PCA components)
loadings = pd.DataFrame(
    pca.components_.T,
    index   = df.columns[:-1],
    columns = ['PC1', 'PC2']
)
print('Feature Loadings (PCA Components):')
print(loadings.round(4).to_string())

Feature Loadings (PCA Components):
                              PC1     PC2
age                        0.4116 -0.2594
annual_spend               0.4215 -0.0333
visits_per_month          -0.4104  0.2083
basket_size                0.4120 -0.1954
days_since_last_visit      0.3786  0.9112
num_categories_purchased   0.4140 -0.1405


### 📊 Interpretation of PC1 and PC2

**PC1 — 'Overall Customer Value Axis' (83.56% variance explained):**
- High positive loadings: `annual_spend` (+0.42), `basket_size` (+0.41), `num_categories_purchased` (+0.41), `age` (+0.41)
- Strong negative loading: `visits_per_month` (–0.41)
- **PC1 separates high-spending, older, infrequent big-basket shoppers** (right) from **young, frequent, small-basket shoppers** (left).
- PC1 alone explains **83.56%** of all variability — it is the dominant axis of customer differentiation.

**PC2 — 'Recency Axis' (5.57% variance explained):**
- Dominant loading: `days_since_last_visit` (+0.91)
- **PC2 separates lapsed customers** (high `days_since_last_visit`, top of plot) from **recently active customers** (bottom).
- This is why Clusters 1 and 3 (both high-value, older) separate vertically — Cluster 1 is lapsed, Cluster 3 is recent.

**Together, 89.13% of total variance is captured** — an excellent 2D representation of 6D customer behaviour.

## Task 5 · Cluster Visualisation — PC1 vs PC2 Scatter Plot — 3 marks

In [9]:
colors = ['#3498DB', '#E67E22', '#27AE60', '#E74C3C']
cluster_names = {
    0: 'Young Frequent Shoppers',
    1: 'Lapsed High-Value Seniors',
    2: 'Core Mid-Market Regulars',
    3: 'Engaged Premium Loyalists'
}

fig, ax = plt.subplots(figsize=(12, 8))

for cid in range(optimal_k):
    mask = df['cluster'] == cid
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors[cid],
        label=f'Cluster {cid}: {cluster_names[cid]}',
        alpha=0.7, s=60, edgecolors='white', linewidth=0.4
    )

# Plot centroids in PCA space
centroids_pca = pca.transform(centroids_scaled)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           c='black', marker='X', s=300, zorder=10, label='Centroids')
for i, (cx, cy) in enumerate(centroids_pca):
    ax.annotate(f'  C{i}', (cx, cy), fontsize=12, fontweight='bold')

ax.set_title('Customer Segments — PCA Visualisation (PC1 vs PC2)',
             fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 — Customer Value Axis ({pca.explained_variance_ratio_[0]*100:.1f}% variance)',
              fontsize=12)
ax.set_ylabel(f'PC2 — Recency Axis ({pca.explained_variance_ratio_[1]*100:.1f}% variance)',
              fontsize=12)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_cluster_visualisation.png', dpi=100, bbox_inches='tight')
plt.show()
print('PCA cluster scatter plot saved.')

PCA cluster scatter plot saved.


### 📊 PCA Visualisation Interpretation
The scatter plot confirms the **4-cluster solution is well-separated and meaningful**:
- **Cluster 0 (Blue — Young Frequent Shoppers)**: Concentrated on the **far left** of PC1 — lowest value axis. Low `days_since_last_visit` places them at the bottom of PC2.
- **Cluster 1 (Orange — Lapsed High-Value Seniors)**: **Far right** on PC1 (high value) and **top** on PC2 (lapsed — high `days_since_last_visit`). These are the most at-risk customers.
- **Cluster 2 (Green — Core Mid-Market)**: Occupies the **centre** — average value, average recency. The largest segment (165 customers).
- **Cluster 3 (Red — Engaged Premium Loyalists)**: **Far right** on PC1 (high value) and **bottom-right** on PC2 (recently active). The most valuable active segment.

The centroid markers (black X) are well-separated in the PCA space, confirming K=4 produces **distinct, non-overlapping clusters** suitable for targeted marketing strategies.